# 01 · 生成可打印 A4 手写采集表（含 ArUco 四角定位与去色网格）

本 Notebook 用于生成标准 A4 纸面手写采集模板（PDF + 预览 PNG + `manifest.json`）：
- 四角内置 OpenCV `DICT_4X4_50`（ID 0~3）视觉定位标记，扫描后自动完成透视校正；
- 格线与提示字使用**淡青色（Cyan）或淡红色（Red）可滤除色**，使用 0.5–0.7 mm 黑色中性笔正常书写后，扫描件可通过色彩通道分离无损滤除网格；
- `manifest.json` 精确记录每个 `(page, row, col)` 与目标字符、书写框毫米坐标的映射关系，供 `02_scan_to_trajectories.ipynb` 确定性切片。

## 1. 挂载 Google Drive 并安装依赖

在 Google Colab 运行时执行以下单元格挂载云端硬盘并安装 `hwfont_sdk` 与 `hwbuilder`。

In [ ]:
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install pymupdf pillow numpy pydantic

# 若已克隆仓库到 /content/inkcraft，直接加入 sys.path
import sys
REPO_ROOT = Path("/content/inkcraft") if IN_COLAB and Path("/content/inkcraft").exists() else Path.cwd().parents[1]
for sub in ("packages/hwfont_sdk", "colab_builder", "backend"):
    p = str(REPO_ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

print("REPO_ROOT:", REPO_ROOT)

## 2. 配置书写者与采集字符集

- 默认 `DEFAULT_CORE_CHARSET` 包含：数字 `0–9`、拉丁字母 `A–Z a–z`、常用中英文标点与数学符号、以及覆盖高频偏旁部首的汉字代表集；
- 强烈建议将 `repeat_ascii=3`，让书写者将数字、英文字母与常用符号手写 3 遍，以获得完全真实的公式/正文同字异形。

In [ ]:
from hwbuilder.sheet_gen import (
    DEFAULT_CORE_CHARSET,
    SheetConfig,
    generate_collection_sheets,
)

WRITER_NAME = "zhangsan"  # 建议英文/拼音标识，便于文件命名
OUT_DIR = Path("/content/drive/MyDrive/inkcraft_builder") / WRITER_NAME / "01_sheets" if IN_COLAB else Path("./work") / WRITER_NAME / "01_sheets"

cfg = SheetConfig(
    writer_name=WRITER_NAME,
    rows=12,
    cols=10,
    grid_color="cyan",      # 可选 "cyan"（推荐）或 "red"
    guide_style="cross",    # 可选 "cross"（田字格）、"mi"（米字格）或 "none"
    preview_dpi=150,
    repeat_ascii=3,         # 数字/字母/标点手写 3 遍
    repeat_cjk=1,           # 汉字样本每字 1 遍（可按需调高）
)

manifest = generate_collection_sheets(
    chars=DEFAULT_CORE_CHARSET,
    out_dir=OUT_DIR,
    config=cfg,
    render_previews=True,
)

print(f"已生成 {len(manifest.pages)} 页采集表 -> {OUT_DIR / manifest.pdf_path}")
print(f"Manifest 路径 -> {OUT_DIR / 'manifest.json'}")

## 3. 预览首页并打印指引

**打印与扫描注意事项**：
1. 彩色打印（确保淡青色/淡红色网格不被转成黑色），打印缩放务必选择 **100% 实际大小**；
2. 请用 **0.5 mm – 0.7 mm 黑色中性笔**自然书写，尽量不超出格子内框；
3. 书写完成后以 **300–600 DPI 彩色扫描**（保存为 `page_000.png`、`page_001.png` ... 或 `.jpg`），放入 `02_scans/` 目录后进入 Notebook 02。

In [ ]:
from PIL import Image
from IPython.display import display

first_preview = OUT_DIR / manifest.pages[0].preview_png
if first_preview.exists():
    img = Image.open(first_preview)
    display(img.resize((600, int(600 * img.height / img.width))))
else:
    print("未找到预览图:", first_preview)